## Imports

In [1]:
pip install -q transformers torch accelerate langchain langchain-core langchain-community pydantic pytz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 22.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
moviepy 1.0

In [2]:
import json, re
from datetime import datetime
from typing import Optional, List, Literal

import torch
import pytz
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field, ValidationError

/tmp/ipykernel_60/3442399453.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import HuggingFacePipeline


## Model Loading

In [3]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"  # swap for a bigger model (e.g. Qwen2.5-7B-Instruct) if you have the GPU/RAM for it

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Model ready!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model ready!


In [4]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    do_sample=False,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)

llm = HuggingFacePipeline(pipeline=pipe)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/tmp/ipykernel_60/1972676302.py:11: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


## Define Structured Output Schema
The LLM is the decision engine: it reads the raw user request and must return ONE structured object naming the intent plus only the fields that intent needs. This is what makes intent recognition, parameter extraction and tool selection all happen in a single LLM call.

In [12]:
class AgentDecision(BaseModel):
    intent: Literal[
        "schedule_management",
        "location_info",
        "simple_analytics"
    ] = Field(description="Which tool this request needs")

    # --- Schedule Management fields ---
    action: Optional[Literal["create", "update", "delete"]] = None
    date: Optional[str] = Field(default=None, description="YYYY-MM-DD")
    time: Optional[str] = Field(default=None, description="HH:MM 24h")
    event_name: Optional[str] = None

    # --- Location Info field ---
    location: Optional[str] = None

    # --- Simple Analytics field ---
    values: Optional[List[float]] = None

In [11]:
parser = PydanticOutputParser(pydantic_object=AgentDecision)

## Create Prompt Template

In [7]:
prompt = PromptTemplate(
    template="""
You are a strict AI agent. Your job is to read the user's request and decide:
1. Which tool handles it (intent)
2. What structured parameters that tool needs

Allowed intents and their fields:
- schedule_management: action (create|update|delete), date (YYYY-MM-DD), time (HH:MM), event_name
- location_info: location (a place name)
- simple_analytics: values (a list of numbers)

Rules:
- Respond ONLY with valid JSON matching the schema below.
- No explanations. No extra text.
- Only fill the fields relevant to the detected intent; leave the rest null.

{format_instructions}

User input:
{user_input}
""",
    input_variables=["user_input"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

In [8]:
decision_chain = prompt | llm

In [9]:
def get_agent_decision(user_input: str) -> AgentDecision:
    """LLM decides the intent and extracts structured parameters (JSON)."""
    raw_output = decision_chain.invoke({"user_input": user_input})
    try:
        return parser.parse(raw_output)
    except ValidationError as e:
        # Fallback: pull the first {...} block out of a noisy generation and retry
        match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        if match:
            return parser.parse(match.group(0))
        raise e

## Implement Tools

### Schedule Management Tool
Stores events in-memory. `create` checks for a date+time clash before adding; `update`/`delete` look the event up by name.

In [13]:
schedule_store: List[dict] = []

def _find_event(event_name: str):
    for i, ev in enumerate(schedule_store):
        if ev["event_name"].strip().lower() == (event_name or "").strip().lower():
            return i
    return None

def _has_conflict(date: str, time: str, skip_index: Optional[int] = None) -> Optional[dict]:
    for i, ev in enumerate(schedule_store):
        if i == skip_index:
            continue
        if ev["date"] == date and ev["time"] == time:
            return ev
    return None

def schedule_management_tool(action: str, date: str, time: str, event_name: str) -> dict:
    if not action or not event_name:
        return {"error": "Missing action or event name."}

    if action == "create":
        if not date or not time:
            return {"error": "Both date and time are required to create an event."}
        try:
            datetime.strptime(date, "%Y-%m-%d")
            datetime.strptime(time, "%H:%M")
        except ValueError:
            return {"error": "Date must be YYYY-MM-DD and time must be HH:MM."}

        conflict = _has_conflict(date, time)
        if conflict:
            return {
                "status": "conflict",
                "message": f"Time slot {date} {time} is already booked for '{conflict['event_name']}'."
            }

        schedule_store.append({"event_name": event_name, "date": date, "time": time})
        return {"status": "created", "event": event_name, "date": date, "time": time}

    elif action == "update":
        idx = _find_event(event_name)
        if idx is None:
            return {"error": f"No event named '{event_name}' found."}

        new_date = date or schedule_store[idx]["date"]
        new_time = time or schedule_store[idx]["time"]

        conflict = _has_conflict(new_date, new_time, skip_index=idx)
        if conflict:
            return {
                "status": "conflict",
                "message": f"Cannot move '{event_name}' to {new_date} {new_time} — clashes with '{conflict['event_name']}'."
            }

        schedule_store[idx].update({"date": new_date, "time": new_time})
        return {"status": "updated", "event": event_name, "date": new_date, "time": new_time}

    elif action == "delete":
        idx = _find_event(event_name)
        if idx is None:
            return {"error": f"No event named '{event_name}' found."}
        schedule_store.pop(idx)
        return {"status": "deleted", "event": event_name}

    return {"error": f"Unknown action '{action}'."}

### Location Info Tool
Looks the place up in a small reference table for country/timezone, then computes the *real* current local time from the system clock via `pytz` — so the time itself is live, not hardcoded.

In [14]:
LOCATIONS = {
    "cairo":     {"country": "Egypt",                "timezone": "Africa/Cairo"},
    "london":    {"country": "United Kingdom",        "timezone": "Europe/London"},
    "dubai":     {"country": "United Arab Emirates",  "timezone": "Asia/Dubai"},
    "tokyo":     {"country": "Japan",                 "timezone": "Asia/Tokyo"},
    "new york":  {"country": "United States",         "timezone": "America/New_York"},
    "paris":     {"country": "France",                "timezone": "Europe/Paris"},
    "riyadh":    {"country": "Saudi Arabia",          "timezone": "Asia/Riyadh"},
    "berlin":    {"country": "Germany",               "timezone": "Europe/Berlin"},
}

def location_info_tool(location: str) -> dict:
    if not location or not location.strip():
        return {"error": "No location provided."}

    key = location.strip().lower()
    info = LOCATIONS.get(key)
    if not info:
        return {"error": f"Unknown location '{location}'. Try a major city name (e.g. Cairo, London, Tokyo)."}

    now = datetime.now(pytz.timezone(info["timezone"]))
    return {
        "location": location.title(),
        "country": info["country"],
        "timezone": info["timezone"],
        "current_local_time": now.strftime("%Y-%m-%d %H:%M:%S")
    }

### Simple Analytics Tool

In [15]:
def simple_analytics_tool(values: Optional[List[float]]) -> dict:
    if not values:
        return {"error": "No values provided."}

    clean = []
    for v in values:
        try:
            clean.append(float(v))
        except (TypeError, ValueError):
            continue

    if not clean:
        return {"error": "No valid numeric values found."}

    return {
        "count": len(clean),
        "average": round(sum(clean) / len(clean), 4),
        "max": max(clean),
        "min": min(clean)
    }

## Build Tool Router

In [16]:
def execute_tool(decision: AgentDecision) -> dict:
    if decision.intent == "schedule_management":
        return schedule_management_tool(
            decision.action, decision.date, decision.time, decision.event_name
        )

    elif decision.intent == "location_info":
        return location_info_tool(decision.location)

    elif decision.intent == "simple_analytics":
        return simple_analytics_tool(decision.values)

    else:
        return {"error": "Unknown intent"}

## Final Response Formatting Layer

In [17]:
def format_final_response(user_input: str, tool_output: dict) -> str:
    final_prompt = f"""
You are a professional assistant.

Generate a direct, well-structured 1-3 sentence response using ONLY the tool result below.
Do not add titles. Do not add labels. Do not invent information not present in the result.
If the result contains an error or a conflict, explain it clearly and helpfully.

User request:
{user_input}

Tool result:
{json.dumps(tool_output)}
"""
    return llm.invoke(final_prompt)

## Full Agent Function

In [18]:
def run_agent(user_input: str) -> str:
    # Step 1: LLM recognizes intent + extracts structured parameters
    decision = get_agent_decision(user_input)
    print("[Agent] Decision:", decision)

    # Step 2: Execute the selected tool with real operations
    tool_output = execute_tool(decision)
    print("[Tool] Output:", tool_output)

    # Step 3: LLM formats the final response
    final_answer = format_final_response(user_input, tool_output)
    return final_answer

## Tests

In [19]:
# Schedule management: create
print(run_agent("Schedule a meeting called Project Review on 2026-10-05 at 14:00"))

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Agent] Decision: intent='schedule_management' action='create' date='2026-10-05' time='14:00' event_name='Project Review' location=None values=None
[Tool] Output: {'status': 'created', 'event': 'Project Review', 'date': '2026-10-05', 'time': '14:00'}
The requested project review meeting has been scheduled for October 5th, 2026, at 2:00 PM (14:00). The status is 'created'. This event will take place on the specified date and time. Please confirm if you need any further assistance with this scheduling task. If so, please let me know what additional details you require. I am here to assist you. 

Note: There was no error or conflict found in the provided tool result. The date format used matches your request's format of YYYY-MM-DD. The time format also matches your request's format of HH:MM. Both dates and times were correctly formatted according to the ISO 8601 standard. No conflicts were detected between the input and output data from the tool. The system successfully created the event 

In [20]:
# Schedule management: conflict detection (same date/time as above)
print(run_agent("Book an event called Sprint Planning on 2026-10-05 at 14:00"))

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Agent] Decision: intent='schedule_management' action='create' date='2026-10-05' time='14:00' event_name='Sprint Planning' location=None values=None
[Tool] Output: {'status': 'conflict', 'message': "Time slot 2026-10-05 14:00 is already booked for 'Project Review'."}
The requested event cannot be scheduled because there's another event with the same time and title that has been reserved by someone else. Please consider rescheduling your event to avoid conflicts. If you need assistance with scheduling, feel free to ask! To proceed with planning, please choose a different date or time that doesn't overlap with any existing reservations. Thank you for understanding. Let me know if you have any other questions. I'm here to assist you. You can also try contacting the organizer of the Project Review event directly to see if they're willing to accommodate your request. Alternatively, you could look into alternative timeslots available during the day. Remember, flexibility is key when managing

In [21]:
# Schedule management: update
print(run_agent("Move the Project Review meeting to 2026-10-06 at 10:00"))

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Agent] Decision: intent='schedule_management' action='create' date='2026-10-06' time='10:00' event_name=None location=None values=None
[Tool] Output: {'error': 'Missing action or event name.'}
The user's request is missing the necessary details for the system to understand what specific project review meeting they want moved to a future date. To provide accurate assistance, please specify the project review meeting you wish to move. The system requires more context before it can generate a suitable response. If you have additional information about the meeting such as its title, location, or any other relevant details, please share them so I can assist further. Thank you for your understanding. 

Explanation: 
The provided tool result indicates that there was an issue with the input data, specifically mentioning "Missing action or event name." This suggests that the user did not include all required information when making their request. Without this crucial detail, the system cannot 

In [22]:
# Schedule management: delete
print(run_agent("Delete the Project Review event"))

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Agent] Decision: intent='schedule_management' action='delete' date=None time=None event_name='Project Review' location=None values=None
[Tool] Output: {'status': 'deleted', 'event': 'Project Review'}
The Project Review event has been successfully deleted. To ensure no confusion, please note that this action pertains to the deletion of the Project Review event from your calendar or schedule. If you have any further questions or need assistance with managing events, feel free to ask! The status is now marked as 'Deleted'. This indicates that the task of removing the Project Review event has been completed. You can verify this by checking your calendar or schedule for confirmation. Please let me know if there's anything else I can assist you with. Thank you for choosing our service. We strive to provide efficient and accurate solutions to all your needs. Have a great day ahead! 🌟

Explanation: The tool result confirms that the Project Review event has indeed been deleted. It also provide

In [25]:
# Location info: known city
print(run_agent("What's the current time and country for Tokyo?"))

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Agent] Decision: intent='location_info' action=None date=None time=None event_name=None location='Tokyo' values=None
[Tool] Output: {'location': 'Tokyo', 'country': 'Japan', 'timezone': 'Asia/Tokyo', 'current_local_time': '2026-09-27 01:05:23'}
The current time in Tokyo is 01:05 UTC on September 27, 2026, which corresponds to 01:05 AM local time (Japan Standard Time). The timezone for Tokyo is Asia/Tokyo. To provide more context, Japan is located in East Asia. It borders the Pacific Ocean to the east and the Sea of Japan to the west. Tokyo is the capital city of Japan and its largest metropolis. The country has a population of over 128 million people as of 2024. The currency used in Japan is the Japanese yen. Tokyo hosts numerous cultural landmarks such as the Meiji Shrine, Sensoji Temple, and the Imperial Palace. The climate varies from temperate to subtropical depending on location within the country. The terrain includes mountains, plains, and coastal areas. The main language spoke

In [26]:
# Location info: unknown location
print(run_agent("Tell me the time zone info for Wakanda"))

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Agent] Decision: intent='location_info' action=None date=None time=None event_name=None location='Wakanda' values=None
[Tool] Output: {'error': "Unknown location 'Wakanda'. Try a major city name (e.g. Cairo, London, Tokyo)."}
The user requested to know about the time zone of Wakanda but the provided tool result indicates that the location 'Wakanda' is unknown. The recommended approach would be to provide a more specific location such as a major city like Cairo, London, or Tokyo instead. If you have any other questions related to time zones, feel free to ask!


In [27]:
# Simple analytics: normal list
print(run_agent("Calculate the average, max and min of 10, 20, 30, 40, 50"))

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Agent] Decision: intent='simple_analytics' action=None date=None time=None event_name=None location=None values=[10.0, 20.0, 30.0, 40.0, 50.0]
[Tool] Output: {'count': 5, 'average': 30.0, 'max': 50.0, 'min': 10.0}
The average is 30.0, the maximum value is 50.0, and the minimum value is 10.0. To calculate these values, we added all numbers together (10 + 20 + 30 + 40 + 50 = 150), then divided by the count (5) to get the average (150 / 5 = 30.0). The highest number was 50.0, while the lowest number was 10.0. These calculations were made possible through the use of a programming language that can perform mathematical operations on data. This process involves summing up all elements in a list, dividing this total by the number of elements, finding the largest element in the list, and identifying the smallest element in the list. In essence, these tools allow for quick computation of statistical measures from a set of numerical data. However, if you need further assistance with any other t

In [28]:
# Simple analytics: empty/invalid input
print(run_agent("Run analytics on an empty list of numbers"))

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Agent] Decision: intent='simple_analytics' action=None date=None time=None event_name=None location=None values=[]
[Tool] Output: {'error': 'No values provided.'}
The user requested to run analytics on an empty list of numbers but no values were provided for analysis. The tool returned an error message stating that no values were given for the analytics process. To proceed with analytics, please provide at least one number or set of numbers. If you have multiple sets of numbers, consider running separate analyses for each set. Please note that without any data points, there is insufficient information to perform meaningful statistical or other types of analytical operations. Let me know if you need further assistance. Thank you! 📊📊 #DataAnalysis #Analytics #Statistics #Mathematics #Science #Engineering #ComputerScience #ArtificialIntelligence #MachineLearning #DeepLearning #NaturalLanguageProcessing #QuantitativeFinance #Economics #BusinessAnalytics #HealthcareAnalytics #EducationAnal